# Customer 360 Reverse ETL to Cosmos DB

This notebook runs end-to-end in a **single Fabric notebook** — no separate deployments needed!

### What it does
1. Reads Wide World Importers sales and customer data from Lakehouse
2. Aggregates transactional data into enriched Customer 360 profiles
3. Generates embeddings on customer profiles using Fabric's built-in OpenAI
4. Writes enriched profiles with embeddings to Cosmos DB (Reverse ETL)

### Getting started
1. **Import** this notebook into your Fabric workspace.
2. **Attach a Lakehouse:** In the Fabric notebook editor, open the **Explorer** tab (left sidebar), click **Add data items**, then select **From OneLake catalog** and choose the Lakehouse that contains the Wide World Importers sample data (`dimension_customer`, `fact_sale`, etc.).
3. **Set your endpoint:** Update `COSMOS_ENDPOINT` in Cell 4 with your Cosmos DB endpoint.
4. **Run** the cells from top to bottom. AAD auth is handled automatically by Fabric.

> **⚠️ IMPORTANT — Cell Language:**  
> After importing, verify that each code cell's language is set to **PySpark**.  
> Imported notebooks typically default to PySpark, but if any cell shows plain Python, click the language dropdown in the bottom-left corner of the code cell and select **PySpark**.

## Section 0: Spark Session Configuration

You need **two JARs** for the Cosmos DB Spark connector with Fabric AAD auth:
1. `azure-cosmos-spark_3-5_2-12` — the Spark connector itself
2. `fabric-cosmos-spark-auth_3` — contains `FabricAccountDataResolver`

> **Important:** `%%configure` must be the very first line of the cell. No comments or blank lines before it.

In [ ]:
%%configure -f
{
    "conf": {
        "spark.jars.packages": "com.azure.cosmos.spark:azure-cosmos-spark_3-5_2-12:4.41.0,com.azure.cosmos.spark:fabric-cosmos-spark-auth_3:1.1.0"
    }
}

## Section 1a: Install Dependencies

This must run **before** the imports in the next cell. After this cell runs, Fabric will reload the Python environment.

In [ ]:
%pip install -U openai azure-cosmos

## Section 1b: Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, max as _max, min as _min, 
    collect_list, concat_ws, lit, datediff, current_date,
    countDistinct, explode, struct, to_json, from_json,
    row_number, desc, when, concat, round as spark_round, to_date,
    broadcast
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DoubleType, ArrayType, TimestampType
)
import json
from datetime import datetime
import os
import openai
from synapse.ml.mlflow import get_mlflow_env_config

# Azure OpenAI Configuration (Fabric Built-in)
# Uses Fabric's AAD token and workload endpoint for authentication
mlflow_env_configs = get_mlflow_env_config()

openai_client = openai.AzureOpenAI(
    azure_endpoint=mlflow_env_configs.workload_endpoint + "cognitive/openai/",
    api_key=mlflow_env_configs.driver_aad_token,
    api_version="2024-02-01",
)

OPENAI_MODEL = "text-embedding-ada-002"
print("✓ Using Fabric's built-in OpenAI embedding model")

# Cosmos DB Configuration
# Replace with your Azure Cosmos DB account endpoints
COSMOS_ENDPOINT = "https://<cosmos-account-name>.documents.azure.com:443/"
COSMOS_DATABASE = "Customer360DB"
COSMOS_CONTAINER = "EnrichedCustomers"

# Get the Azure AD tenant ID from the AAD token (for reference/debugging)
import base64, json as _json
_token_parts = mlflow_env_configs.driver_aad_token.split(".")
_payload = _token_parts[1] + "=" * (4 - len(_token_parts[1]) % 4)  # fix padding
TENANT_ID = _json.loads(base64.b64decode(_payload))["tid"]
print(f"✓ Tenant ID: {TENANT_ID}")
print("✓ Configuration loaded")

## Section 2: Read Data from Lakehouse

In [ ]:
print("\n" + "="*80)
print("SECTION 2: Reading data from Lakehouse Tables")
print("="*80)

# Read dimension tables
dim_customer = spark.read.table("dimension_customer")
dim_city = spark.read.table("dimension_city")
dim_stock_item = spark.read.table("dimension_stock_item")

# Read fact table
fact_sale = spark.read.table("fact_sale")

print(f"\u2713 Customers: {dim_customer.count():,} rows")
print(f"\u2713 Sales Transactions: {fact_sale.count():,} rows")
print(f"\u2713 Products: {dim_stock_item.count():,} rows")
print(f"\u2713 Cities: {dim_city.count():,} rows")

# Display schemas to verify column names
print("\n" + "="*80)
print("VERIFYING SCHEMAS")
print("="*80)
print("\nFact Sale Schema:")
fact_sale.printSchema()
print("\nDimension Customer Schema:")
dim_customer.printSchema()

## Section 3: Build Customer 360 Aggregations

In [ ]:
print("\n" + "="*80)
print("SECTION 3: Building Customer 360 Profiles")
print("="*80)

# ── Exclude synthetic "Unknown" customer ─────────────────────────────────────
# The Fabric WWI sample dataset contains ~17 million Fact.Sale rows attributed
# to an "Unknown" placeholder customer (CustomerKey = 0).  These are NOT real
# business transactions.  They originate from SQL Server's stored procedure
# Configuration_PopulateLargeSaleTable, which bulk-inserts millions of synthetic
# rows into Fact.Sale for columnstore-index and query-performance benchmarking.
# Because the rows have no real customer context, the ETL assigns them to the
# standard dimensional-modeling "Unknown" member.
#
# We filter BEFORE any joins so these ~17M rows are excluded from purchase
# aggregations, product-preference window functions, percentile-based
# segmentation, embeddings, and Cosmos DB writes — not processed and thrown
# away at the end.
# ─────────────────────────────────────────────────────────────────────────────
dim_customer = dim_customer.filter(col("Customer") != "Unknown")
fact_sale = fact_sale.join(
    broadcast(dim_customer.select("CustomerKey")), "CustomerKey", "leftsemi"
)
print("  Excluded 'Unknown' placeholder customer (synthetic SQL Server benchmark data)")

# 3.1: Customer Purchase History
print("\nAggregating purchase history...")
customer_purchases = fact_sale.join(
    dim_customer, 
    "CustomerKey", 
    "inner"
).groupBy(
    "CustomerKey",
    "Customer"
).agg(
    count("*").alias("total_transactions"),
    _sum("Quantity").alias("total_items_purchased"),
    _sum("TotalExcludingTax").alias("total_revenue"),
    avg("TotalExcludingTax").alias("avg_transaction_value"),
    _max("InvoiceDateKey").alias("last_purchase_date_key"),
    _min("InvoiceDateKey").alias("first_purchase_date_key"),
    countDistinct("StockItemKey").alias("unique_products_purchased")
)

print(f"✓ Customer purchase aggregations: {customer_purchases.count():,} customers")

# Resolve InvoiceDateKey to actual calendar dates
# Handles DateType (use directly), IntegerType in YYYYMMDD format, or other integers
from pyspark.sql.types import DateType as _DateType, TimestampType as _TsType
_idk_type = fact_sale.schema["InvoiceDateKey"].dataType
print(f"  InvoiceDateKey column type: {_idk_type}")

if isinstance(_idk_type, (_DateType, _TsType)):
    # Already a date or timestamp — use directly
    customer_purchases = customer_purchases.withColumn(
        "last_purchase_date", col("last_purchase_date_key").cast("date")
    ).withColumn(
        "first_purchase_date", col("first_purchase_date_key").cast("date")
    )
else:
    # Integer key — check if YYYYMMDD format (values > 19000101) vs epoch days
    _max_key = customer_purchases.agg(_max("last_purchase_date_key")).collect()[0][0]
    if _max_key is not None and _max_key > 19000101:
        print(f"  → Detected YYYYMMDD integer format (max={_max_key})")
        customer_purchases = customer_purchases.withColumn(
            "last_purchase_date", to_date(col("last_purchase_date_key").cast("string"), "yyyyMMdd")
        ).withColumn(
            "first_purchase_date", to_date(col("first_purchase_date_key").cast("string"), "yyyyMMdd")
        )
    else:
        print(f"  → Small integer key (max={_max_key}), interpreting as epoch days")
        customer_purchases = customer_purchases.withColumn(
            "last_purchase_date", col("last_purchase_date_key").cast("date")
        ).withColumn(
            "first_purchase_date", col("first_purchase_date_key").cast("date")
        )

print("  Sample resolved dates:")
customer_purchases.select("Customer", "last_purchase_date", "first_purchase_date").show(3, truncate=False)

# 3.2: Product Preferences (Top 5 products per customer)
print("\nIdentifying product preferences...")
customer_product_prefs = fact_sale.join(
    dim_stock_item,
    "StockItemKey",
    "inner"
).groupBy(
    fact_sale["CustomerKey"],
    dim_stock_item["StockItem"]
).agg(
    _sum("Quantity").alias("product_quantity"),
    _sum("TotalExcludingTax").alias("product_revenue")
).withColumn(
    "rank",
    row_number().over(
        Window.partitionBy("CustomerKey").orderBy(desc("product_revenue"))
    )
).filter(col("rank") <= 5)\
 .groupBy("CustomerKey")\
 .agg(
     collect_list(
         struct(
             col("StockItem").alias("product"),
             col("product_quantity").alias("quantity"),
             col("product_revenue").alias("revenue")
         )
     ).alias("top_products")
 )

print(f"✓ Product preferences identified for {customer_product_prefs.count():,} customers")

# 3.3: Customer Segment (based on revenue percentiles)
# Use percentile-based thresholds so the split adapts to any dataset's revenue range
print("\nCalculating customer segments...")
_p60, _p30 = customer_purchases.approxQuantile("total_revenue", [0.60, 0.30], 0.01)
print(f"  Revenue percentile thresholds: High >= ${_p60:,.0f}, Medium >= ${_p30:,.0f}")

customer_purchases_with_segment = customer_purchases.withColumn(
    "customer_segment",
    when(col("total_revenue") >= _p60, "High Value")
    .when(col("total_revenue") >= _p30, "Medium Value")
    .otherwise("Low Value")
).withColumn(
    "avg_days_between_purchases",
    datediff(
        current_date(),
        col("last_purchase_date")
    ) / col("total_transactions")
)

# 3.4: Recency Metrics
print("\nCalculating recency metrics...")
customer_recency = customer_purchases_with_segment.withColumn(
    "days_since_last_purchase",
    datediff(
        current_date(),
        col("last_purchase_date")
    )
).withColumn(
    "customer_lifetime_days",
    datediff(
        col("last_purchase_date"),
        col("first_purchase_date")
    )
)

# 3.5: Join all customer data
print("\nJoining all customer dimensions...")

customer_360 = dim_customer.join(
    customer_recency.drop("Customer"),
    "CustomerKey",
    "left"
).join(
    customer_product_prefs,
    "CustomerKey",
    "left"
).select(
    col("CustomerKey").alias("customer_id"),
    col("Customer").alias("customer_name"),
    col("BuyingGroup").alias("buying_group"),
    col("Category").alias("category"),
    col("PostalCode").alias("postal_code"),
    col("total_transactions"),
    col("total_items_purchased"),
    spark_round(col("total_revenue"), 2).alias("total_revenue"),
    spark_round(col("avg_transaction_value"), 2).alias("avg_transaction_value"),
    col("unique_products_purchased"),
    col("customer_segment"),
    col("last_purchase_date").cast("string").alias("last_purchase_date"),
    col("first_purchase_date").cast("string").alias("first_purchase_date"),
    col("days_since_last_purchase"),
    col("customer_lifetime_days"),
    spark_round(col("avg_days_between_purchases"), 1).alias("avg_days_between_purchases"),
    col("top_products")
)

print(f"✓ Customer 360 profiles built: {customer_360.count():,} customers")

# Quick segment distribution check
_seg_rows = customer_360.groupBy("customer_segment").agg(
    count("*").alias("customers"),
    spark_round(avg("total_revenue"), 2).alias("avg_revenue")
).orderBy(desc("customers"))
print("\nSegment distribution:")
_seg_rows.show(truncate=False)

## Section 4: Create Text Profiles for Embeddings

In [ ]:
print("\n" + "="*80)
print("SECTION 4: Creating Text Profiles for Embedding Generation")
print("="*80)

# Convert top products array to readable string
def format_top_products(top_products):
    if top_products is None:
        return "No purchase history"
    products_list = [f"{p['product']} (${p['revenue']:.0f})" for p in top_products[:3]]
    return ", ".join(products_list)

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

format_products_udf = udf(format_top_products, StringType())

# Create rich text profile for each customer
# Lead with segment to give it strong embedding weight
customer_360_with_text = customer_360.withColumn(
    "customer_profile_text",
    concat_ws(" | ",
        concat(col("customer_segment"), lit(" Customer")),
        concat(lit("Customer: "), col("customer_name")),
        concat(lit("Category: "), col("category")),
        concat(lit("Buying Group: "), col("buying_group")),
        concat(lit("Segment: "), col("customer_segment")),
        concat(lit("Total Purchases: "), col("total_transactions")),
        concat(lit("Lifetime Revenue: $"), col("total_revenue")),
        concat(lit("Avg Transaction: $"), col("avg_transaction_value")),
        concat(lit("Days Since Last Purchase: "), col("days_since_last_purchase")),
        concat(lit("Top Products: "), format_products_udf(col("top_products")))
    )
)

print("\u2713 Customer profile text created")
customer_360_with_text.select("customer_id", "customer_name", "customer_profile_text").show(3, truncate=100)

## Section 5: Generate Embeddings Using Fabric's Built-in OpenAI

**Approach 1 (Active):** Pandas-based — simple and good for datasets up to ~1,000 customers.  
**Approach 2 (Commented out):** Spark UDF — distributed processing for 10K+ customers.

In [ ]:
print("\n" + "="*80)
print("SECTION 5: Generating Embeddings with Fabric's Built-in OpenAI Model")
print("="*80)

def generate_embeddings(text):
    """Generate embeddings using Fabric's built-in OpenAI model."""
    try:
        response = openai_client.embeddings.create(
            input=text,
            model=OPENAI_MODEL
        )
        embeddings = response.model_dump()
        return embeddings['data'][0]['embedding']
    except Exception as e:
        print(f"Error generating embedding: {e}")
        return None

def get_embeddings_batch(texts_list):
    """Generate embeddings for a batch of texts."""
    embeddings = []
    for i, text in enumerate(texts_list):
        if i % 10 == 0:
            print(f"Processing embedding {i+1}/{len(texts_list)}...")
        embedding = generate_embeddings(text.strip())
        embeddings.append(embedding)
    return embeddings

# Test the embedding model first
print("\nTesting Fabric's OpenAI embedding model...")
test_text = "Customer 360 profile for semantic search"
test_embedding = generate_embeddings(test_text.strip())
print(f"\u2713 Embedding model working! Generated {len(test_embedding)}-dimensional vector")

# --- APPROACH 1: Pandas-based (recommended for < 1000 rows) ---
print("\n" + "-"*80)
print("Using Pandas for embedding generation (recommended for <1000 rows)")
print("-"*80)

print("\nCollecting customer profiles for embedding generation...")
customer_profiles_pd = customer_360_with_text.select(
    "customer_id",
    "customer_profile_text"
).toPandas()

print(f"Generating embeddings for {len(customer_profiles_pd)} customers...")
print("This may take 1-2 minutes...")
customer_profiles_pd["embedding"] = get_embeddings_batch(
    customer_profiles_pd["customer_profile_text"].tolist()
)

print("\u2713 Embeddings generated successfully using Fabric's built-in model")

# Convert back to Spark DataFrame
from pyspark.sql.types import ArrayType, FloatType

embedding_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_profile_text", StringType(), False),
    StructField("embedding", ArrayType(FloatType()), False)
])

embeddings_df = spark.createDataFrame(customer_profiles_pd, schema=embedding_schema)

# Join embeddings back to full customer profile
customer_360_final = customer_360.join(
    embeddings_df.select("customer_id", "embedding"),
    "customer_id",
    "inner"
)

print(f"\u2713 Final Customer 360 with embeddings: {customer_360_final.count():,} customers")

### (Optional) Approach 2: Distributed Spark UDF

Uncomment and use this cell instead of the Pandas approach above for production workloads with 10K+ customers.

In [ ]:
# --- APPROACH 2: Distributed Spark UDF (uncomment for large datasets) ---

# from pyspark.sql.functions import udf
# from pyspark.sql.types import ArrayType, FloatType
#
# @udf(returnType=ArrayType(FloatType()))
# def generate_embedding_udf(text):
#     if text is None or text.strip() == "":
#         return None
#     try:
#         response = openai.embeddings.create(
#             input=text.strip(), data    
            
#             model=OPENAI_MODEL
#         )
#         embeddings = response.model_dump()
#         return embeddings['data'][0]['embedding']
#     except Exception as e:
#         print(f"Error in UDF: {e}")
#         return None
#
# customer_360_with_embeddings = customer_360_with_text.withColumn(
#     "embedding",
#     generate_embedding_udf(col("customer_profile_text"))
# )
#
# customer_360_final = customer_360_with_embeddings.filter(
#     col("embedding").isNotNull()
# ).drop("customer_profile_text")
#
# print(f"Generated embeddings for {customer_360_final.count():,} customers")

## Section 6: Prepare Data for Cosmos DB

In [ ]:
print("\n" + "="*80)
print("SECTION 6: Preparing Data for Cosmos DB")
print("="*80)

# Cosmos DB requires an 'id' field and partition key
cosmos_ready_df = customer_360_final.withColumn(
    "id", col("customer_id").cast("string")
).select(
    col("id"),
    col("customer_id"),
    col("customer_name"),
    col("category"),
    col("buying_group"),
    col("postal_code"),
    col("total_transactions"),
    col("total_items_purchased"),
    col("total_revenue"),
    col("avg_transaction_value"),
    col("unique_products_purchased"),
    col("customer_segment"),
    col("last_purchase_date"),
    col("first_purchase_date"),
    col("days_since_last_purchase"),
    col("customer_lifetime_days"),
    col("avg_days_between_purchases"),
    col("top_products"),
    col("embedding")
).withColumn(
    "last_updated", lit(datetime.now().isoformat())
)

print("\u2713 Data formatted for Cosmos DB")
cosmos_ready_df.printSchema()

## Section 7: Reverse ETL — Write to Cosmos DB

Uses the Cosmos DB Spark Connector with Fabric AAD auth.  
Requires both JARs loaded via `%%configure` in Cell 2:
- `azure-cosmos-spark_3-5_2-12` (Spark connector)
- `fabric-cosmos-spark-auth_3` (`FabricAccountDataResolver`)

This version uses the **Python SDK** to create the database/container with proper vector search policies (vector embedding policy and vector indexing policy cannot be configured via Cosmos Catalog SQL).

In [ ]:
print("\n" + "="*80)
print("SECTION 7: Reverse ETL - Writing to Cosmos DB")
print("="*80)

# Configure Cosmos DB Spark connection using Fabric AAD auth
cosmos_config = {
    "spark.cosmos.accountEndpoint": COSMOS_ENDPOINT,
    "spark.cosmos.account.tenantId": TENANT_ID,
    "spark.cosmos.accountDataResolverServiceName": "com.azure.cosmos.spark.fabric.FabricAccountDataResolver",
    "spark.cosmos.auth.type": "AccessToken",
    "spark.cosmos.useGatewayMode": "true",
    "spark.cosmos.database": COSMOS_DATABASE,
    "spark.cosmos.container": COSMOS_CONTAINER,
    "spark.cosmos.write.strategy": "ItemOverwrite",
    "spark.cosmos.write.bulk.enabled": "true"
}

print(f"Writing {cosmos_ready_df.count()} customer profiles to Cosmos DB...")
print(f"Database: {COSMOS_DATABASE}")
print(f"Container: {COSMOS_CONTAINER}")

# Create database and container with vector search policies using Python SDK
import time as _time
from azure.cosmos import CosmosClient, PartitionKey
from azure.core.credentials import AccessToken as _AccessToken

class _FabricCosmosCredential:
    """Gets an AAD token scoped to Cosmos DB via Fabric runtime.

    Tries mssparkutils first, then notebookutils, then the JVM
    TokenLibrary directly (needed after %pip install reloads the
    Python environment and Fabric doesn't re-inject the wrappers).
    """
    def get_token(self, *scopes, **kwargs):
        try:
            token = mssparkutils.credentials.getToken("https://cosmos.azure.com/")
        except NameError:
            try:
                token = notebookutils.credentials.getToken("https://cosmos.azure.com/")
            except NameError:
                token = (spark._jvm.com.microsoft.azure.synapse
                         .tokenlibrary.TokenLibrary
                         .getAccessToken("https://cosmos.azure.com/"))
        return _AccessToken(token, int(_time.time()) + 3600)

try:
    _cosmos_client = CosmosClient(COSMOS_ENDPOINT, credential=_FabricCosmosCredential())
    _database = _cosmos_client.create_database_if_not_exists(id=COSMOS_DATABASE)
    print(f"\u2713 Database '{COSMOS_DATABASE}' verified/created")

    # Vector embedding policy — tells Cosmos DB about the embedding field
    _vector_embedding_policy = {
        "vectorEmbeddings": [
            {
                "path": "/embedding",
                "dataType": "float32",
                "distanceFunction": "cosine",
                "dimensions": 1536
            }
        ]
    }

    # Indexing policy — adds a vector index and excludes raw embedding from range index
    _indexing_policy = {
        "indexingMode": "consistent",
        "automatic": True,
        "includedPaths": [{"path": "/*"}],
        "excludedPaths": [
            {"path": "/embedding/*"},
            {"path": "/\"_etag\"/?"}
        ],
        "vectorIndexes": [
            {
                "path": "/embedding",
                "type": "quantizedFlat"
            }
        ]
    }

    from azure.cosmos import ThroughputProperties
    _container = _database.create_container_if_not_exists(
        id=COSMOS_CONTAINER,
        partition_key=PartitionKey(path="/customer_id"),
        indexing_policy=_indexing_policy,
        vector_embedding_policy=_vector_embedding_policy,
        offer_throughput=ThroughputProperties(auto_scale_max_throughput=5000)
    )
    print(f"\u2713 Container '{COSMOS_CONTAINER}' verified/created with vector search policies")
    print(f"  - Vector path: /embedding (1536 dims, cosine, float32)")
    print(f"  - Vector index type: quantizedFlat")
except Exception as e:
    print(f"Note: Container setup issue (may already exist): {e}")

# Write data to Cosmos DB via Spark connector
try:
    cosmos_ready_df.write \
        .format("cosmos.oltp") \
        .options(**cosmos_config) \
        .mode("append") \
        .save()
    
    print("\u2713 Successfully wrote customer data to Cosmos DB!")
    print(f"\u2713 Reverse ETL Complete - {cosmos_ready_df.count()} enriched customer profiles available in Cosmos DB")
except Exception as e:
    print(f"Error writing to Cosmos DB: {e}")

## Section 8: Save Enriched Data to Lakehouse (Gold Layer)

In [ ]:
print("\n" + "="*80)
print("SECTION 8: Saving Enriched Profiles to Lakehouse (Gold Layer)")
print("="*80)

cosmos_ready_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_customer_360_enriched")

print("\u2713 Customer 360 profiles saved to Lakehouse table: gold_customer_360_enriched")

## Section 9: Summary Statistics

In [ ]:
print("\n" + "="*80)
print("DEMO SUMMARY")
print("="*80)
print(f"\u2713 Total Customers Processed: {customer_360_final.count():,}")
print(f"\u2713 Embeddings Generated: {embeddings_df.count():,}")
print(f"\u2713 Records Written to Cosmos DB: {cosmos_ready_df.count():,}")
print("\nCustomer Segment Distribution:")
customer_360_final.groupBy("customer_segment").count().orderBy(desc("count")).show()

print("\n\u2713 Pipeline Complete!")
print("\nNext Steps:")
print("1. Query Cosmos DB using vector search for similar customers")
print("2. Build a web app with semantic search capabilities")
print("3. Use customer profiles for personalization and recommendations")
print("="*80)

## Section 10: Cosmos DB vs Lakehouse — Query Latency Benchmark

This cell demonstrates the speed advantage of Azure Cosmos DB for **operational queries** versus reading from the Lakehouse via Spark SQL.

- **Cosmos DB (Python SDK):** Direct point-read and query over the NoSQL API — optimized for low-latency, application-serving workloads.
- **Lakehouse (Spark SQL):** Distributed query over Delta tables — optimized for large-scale analytical processing, not sub-second lookups.

The benchmark runs identical filter queries against both backends and visualizes the latency difference.

In [ ]:
import time
import statistics
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from azure.cosmos import CosmosClient, PartitionKey
import time as _time
from collections import namedtuple

# ── Cosmos DB client (reuse credential from Section 7) ──
_AccessToken = namedtuple("_AccessToken", ["token", "expires_on"])

class _FabricCosmosCredential:
    def get_token(self, *scopes, **kwargs):
        try:
            token = mssparkutils.credentials.getToken("https://cosmos.azure.com/")
        except NameError:
            try:
                token = notebookutils.credentials.getToken("https://cosmos.azure.com/")
            except NameError:
                token = (spark._jvm.com.microsoft.azure.synapse
                         .tokenlibrary.TokenLibrary
                         .getAccessToken("https://cosmos.azure.com/"))
        return _AccessToken(token, int(_time.time()) + 3600)

cosmos_client = CosmosClient(COSMOS_ENDPOINT, credential=_FabricCosmosCredential())
container = cosmos_client.get_database_client(COSMOS_DATABASE).get_container_client(COSMOS_CONTAINER)

# ── Benchmark parameters ──
NUM_ITERATIONS = 5
QUERY_FILTER = "High Value"  # Filter by customer_segment

# ── Warmup (1 call each to avoid measuring cold-start) ──
print("Warming up...")
_ = list(container.query_items(
    query="SELECT TOP 1 c.customer_name FROM c WHERE c.customer_segment = @seg",
    parameters=[{"name": "@seg", "value": QUERY_FILTER}],
    enable_cross_partition_query=True
))
spark.sql(f"SELECT customer_name FROM gold_customer_360_enriched WHERE customer_segment = '{QUERY_FILTER}' LIMIT 1").collect()
print("Warmup complete.\n")

# ── Benchmark: Cosmos DB (Python SDK) ──
cosmos_times = []
cosmos_row_count = 0
for i in range(NUM_ITERATIONS):
    t0 = time.perf_counter()
    results = list(container.query_items(
        query="SELECT c.customer_name, c.customer_segment, c.total_revenue, c.avg_transaction_value, c.total_transactions FROM c WHERE c.customer_segment = @seg",
        parameters=[{"name": "@seg", "value": QUERY_FILTER}],
        enable_cross_partition_query=True
    ))
    elapsed = (time.perf_counter() - t0) * 1000
    cosmos_times.append(elapsed)
    cosmos_row_count = len(results)

# ── Benchmark: Lakehouse (Spark SQL) ──
spark_times = []
spark_row_count = 0
for i in range(NUM_ITERATIONS):
    t0 = time.perf_counter()
    rows = spark.sql(f"""
        SELECT customer_name, customer_segment, total_revenue, avg_transaction_value, total_transactions
        FROM gold_customer_360_enriched
        WHERE customer_segment = '{QUERY_FILTER}'
    """).collect()
    elapsed = (time.perf_counter() - t0) * 1000
    spark_times.append(elapsed)
    spark_row_count = len(rows)

# ── Results ──
cosmos_avg = statistics.mean(cosmos_times)
cosmos_p50 = statistics.median(cosmos_times)
spark_avg  = statistics.mean(spark_times)
spark_p50  = statistics.median(spark_times)
speedup    = spark_avg / cosmos_avg if cosmos_avg > 0 else float("inf")

print("=" * 70)
print("QUERY LATENCY BENCHMARK — Cosmos DB vs Lakehouse (Spark SQL)")
print("=" * 70)
print(f"Query: SELECT ... WHERE customer_segment = '{QUERY_FILTER}'")
print(f"Iterations: {NUM_ITERATIONS}  |  Rows returned: {cosmos_row_count}")
print("-" * 70)
print(f"{'Backend':<25} {'Avg (ms)':>10} {'Median (ms)':>12} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 70)
print(f"{'Cosmos DB (SDK)':<25} {cosmos_avg:>10.1f} {cosmos_p50:>12.1f} {min(cosmos_times):>10.1f} {max(cosmos_times):>10.1f}")
print(f"{'Lakehouse (Spark SQL)':<25} {spark_avg:>10.1f} {spark_p50:>12.1f} {min(spark_times):>10.1f} {max(spark_times):>10.1f}")
print("-" * 70)
print(f"⚡ Cosmos DB is ~{speedup:.0f}x faster on average for this operational query")
print("=" * 70)

# ── Bar chart ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Average latency comparison
backends = ["Cosmos DB\n(Python SDK)", "Lakehouse\n(Spark SQL)"]
avgs = [cosmos_avg, spark_avg]
colors = ["#0078D4", "#742774"]
bars = ax1.bar(backends, avgs, color=colors, width=0.5, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, avgs):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(avgs) * 0.02,
             f"{val:.0f} ms", ha="center", va="bottom", fontweight="bold", fontsize=13)
ax1.set_ylabel("Latency (ms)", fontsize=12)
ax1.set_title(f"Average Query Latency ({NUM_ITERATIONS} runs)", fontsize=14, fontweight="bold")
ax1.spines[["top", "right"]].set_visible(False)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

# Right: All iterations line chart
ax2.plot(range(1, NUM_ITERATIONS + 1), cosmos_times, "o-", color="#0078D4", label="Cosmos DB (SDK)", linewidth=2, markersize=7)
ax2.plot(range(1, NUM_ITERATIONS + 1), spark_times, "s-", color="#742774", label="Lakehouse (Spark SQL)", linewidth=2, markersize=7)
ax2.set_xlabel("Iteration", fontsize=12)
ax2.set_ylabel("Latency (ms)", fontsize=12)
ax2.set_title("Per-Iteration Latency", fontsize=14, fontweight="bold")
ax2.legend(fontsize=11)
ax2.spines[["top", "right"]].set_visible(False)
ax2.set_xticks(range(1, NUM_ITERATIONS + 1))

fig.suptitle(f"Cosmos DB vs Lakehouse — {speedup:.0f}x faster for operational queries", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()